In [1]:
!pip install chromadb
!pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
#LOAD EMBEDDING MODEL

In [3]:
embedding_model = SentenceTransformer(

    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Personal\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Personal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
#CREATE TELECOM KNOWLEDGE BASE

In [4]:
documents = [

    "Broadband installation may take 3 to 5 business days.",

    "Late payment after 7 days may suspend telecom services.",

    "SIM activation may take up to 24 hours.",

    "Roaming activation usually takes 1 hour.",

    "eSIM activation failure can be resolved by rescanning QR code.",

    "Router restart can fix broadband connectivity issues.",

    "Mobile app login failures may require password reset.",

    "Refund requests are processed within 7 business days.",

    "International roaming charges depend on destination country.",

    "Network signal issues may occur inside buildings."
]

In [ ]:
#CREATE CHROMADB DATABASE

In [5]:
client = chromadb.Client()

collection = client.create_collection(

    name="telecom_knowledge_base"
)

print("ChromaDB Collection Created")

ChromaDB Collection Created


In [ ]:
#CREATE EMBEDDINGS

In [6]:
embeddings = embedding_model.encode(
    documents
)

print("Embeddings Created")

Embeddings Created


In [ ]:
#STORE DOCUMENTS

In [7]:
for i, doc in enumerate(documents):

    collection.add(

        documents=[doc],

        embeddings=[
            embeddings[i].tolist()
        ],

        ids=[str(i)]
    )

print("Documents Stored")

Documents Stored


In [ ]:
#CREATE RETRIEVAL FUNCTION

In [8]:
def retrieve_context(query):

    query_embedding = embedding_model.encode(
        query
    )

    results = collection.query(

        query_embeddings=[
            query_embedding.tolist()
        ],

        n_results=2
    )

    return results["documents"][0]

In [ ]:
#TEST RAG SYSTEM

In [9]:
query = "Why is my SIM activation delayed?"

results = retrieve_context(query)

print(results)

['SIM activation may take up to 24 hours.', 'Roaming activation usually takes 1 hour.']


In [ ]:
#CREATE RESPONSE FUNCTION

In [10]:
def generate_response(user_query):

    context = retrieve_context(
        user_query
    )

    response = f"""

Customer Query:
{user_query}

Relevant Telecom Information:
{context}

AI Response:
We understand your concern.
Based on our telecom knowledge base,
the issue may be related to service
activation or temporary processing delay.

Our support team recommends waiting
for the expected activation duration.
If the issue still continues, please
contact customer support for further
assistance.

"""

    return response

In [ ]:
#TEST AI RESPONSE

In [11]:
query = "My SIM activation is still pending"

response = generate_response(query)

print(response)



Customer Query:
My SIM activation is still pending

Relevant Telecom Information:
['SIM activation may take up to 24 hours.', 'Roaming activation usually takes 1 hour.']

AI Response:
We understand your concern.
Based on our telecom knowledge base,
the issue may be related to service
activation or temporary processing delay.

Our support team recommends waiting
for the expected activation duration.
If the issue still continues, please
contact customer support for further
assistance.


